# generator-loss-fool-discriminator — ex2: logits-form non-saturating G loss via BCE-with-logits

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `generator-loss-fool-discriminator`. Running the final beacon cell reports progress against the `GAN: Generator loss to fool D` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Generator loss to fool D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`generator-loss-fool-discriminator`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "generator-loss-fool-discriminator"
DD_SUBTOPIC = "GAN: Generator loss to fool D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Generator loss — logits form (numerically stable)

Ex1 used `F.binary_cross_entropy(d_pred_prob, ones_like)` — the bare-probability form. In production, D's last layer is typically a Linear / Conv whose OUTPUT is a logit; sigmoid is fused into the loss:

```python
d_logits = D(fakes)                              # raw, no sigmoid
loss_G   = F.binary_cross_entropy_with_logits(
    d_logits, t.ones_like(d_logits)
)
```

**Why `_with_logits`.** Numerically stable. `log(sigmoid(z))` and `log(1 - sigmoid(z))` both lose precision when `|z|` is large; the fused form uses `log1p(exp(-z))` and stays finite up to `|z| ≈ 80`.

**Same target=1 trick as ex1.** The non-saturating G loss still uses target 1 on fakes — only the loss FUNCTION changes (logits vs probs).

### Exercise 2 — logits-form non-saturating G loss via BCE-with-logits

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `F.binary_cross_entropy_with_logits` with `t.ones_like` targets to compute the numerically-stable generator loss on D's logits (pre-sigmoid).
> Keywords: gan, generator-loss, bce-logits, numerical-stability
> ```

**KCs targeted:** `bce-with-logits-form`, `logit-vs-probability-domain`

Implement `ex2_generator_loss_from_logits(d_logits)`. The numerically-stable form of ex1's G loss — now consuming D's logits directly:

1. `d_logits` is D's pre-sigmoid output on the fake batch, shape `(B,)`, values in `(-inf, +inf)`.
2. Build targets: `targets = t.ones_like(d_logits)`.
3. Return `F.binary_cross_entropy_with_logits(d_logits, targets)` — a scalar.

Why this version. `binary_cross_entropy_with_logits` fuses sigmoid + BCE and uses `log1p(exp(-z))` internally. It stays finite for `|z| < ~80`, whereas the bare `binary_cross_entropy(sigmoid(z), 1)` blows up at `|z| > ~16` due to log(0).

Input: `d_logits` — `(B,)` float tensor.
Output: scalar tensor.

The visualization plots G loss as a function of the logit value, with the bare-prob form overlaid — showing the numerical divergence at large positive/negative logits.

In [ ]:
def ex2_generator_loss_from_logits(d_logits: Tensor) -> Tensor:
    import torch.nn.functional as F
    targets = t.ones_like(d_logits)
    return F.binary_cross_entropy_with_logits(d_logits, targets)


<details><summary>Solution</summary>

```python
def ex2_generator_loss_from_logits(d_logits: Tensor) -> Tensor:
    import torch.nn.functional as F
    targets = t.ones_like(d_logits)
    return F.binary_cross_entropy_with_logits(d_logits, targets)
```

**Why the logit form is the production default.** In ex1, `d_pred = sigmoid(linear_output)` then `BCE(d_pred, 1)`. Two places to lose precision: `sigmoid` flushes large negatives to 0, then `log(0) = -inf`. Fusing the two sidesteps the intermediate and uses `log1p(exp(-z))` — finite for any `|z| < ~80`.

**Discriminator output should be a logit, not a probability.** Drop the `nn.Sigmoid()` from D's last layer when you switch to `_with_logits`. Forgetting and sigmoid-ing twice is a silent bug: D's effective output is `sigmoid(sigmoid(x))`, training works but slower.

**Same target=1 trick.** The G→fool-D semantic is unchanged. Only the loss function and the meaning of its first arg shift.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()